In [31]:
import json
from pathlib import Path

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError


In [ ]:
url = (
    r"https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.jsonlKVP"
    r"?CatalogNumber,ScientificName,latitude,longitude,DepthInMeters,LargeMarineEcosystem,Country,ObservationDate,ObservationYear"
    r"&ShallowFlag=0"
    r"&ObservationYear%3E=2018"
    r"&Country=%22USA%22"
    r"&LargeMarineEcosystem=~%22%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29%22"
)

In [33]:
class DeepSeaCoralRecord(BaseModel):
    model_config = ConfigDict(populate_by_name=True, frozen=True)

    latitude: float
    longitude: float
    catalog_num: int = Field(alias="CatalogNumber")
    scientific_name: str | None = Field(default=None, alias="ScientificName")
    depth_m: float | None = Field(default=None, alias="DepthInMeters")
    large_marine_ecosystem: str = Field(alias="LargeMarineEcosystem")
    country: str = Field(alias="Country")
    observation_date: str = Field(alias="ObservationDate")
    observation_year: int = Field(alias="ObservationYear")

In [34]:
def download(url: str, dest: Path) -> None:
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            f.writelines(r.iter_content(1 << 20))

In [35]:
def read_records(file_path: Path) -> list[DeepSeaCoralRecord]:
    records: list[DeepSeaCoralRecord] = []
    with open(file_path) as f:
        for line in f:
            try:
                record = DeepSeaCoralRecord(**json.loads(line))
                records.append(record)
            except ValidationError:
                continue
    return records

In [36]:
dest = Path("../data/raw/deep_sea_corals_se.jsonl")
dest.parent.mkdir(exist_ok=True)

download(url, dest)
records = read_records(dest)

print(f"Validated: {len(records)} records")

Validated: 145137 records


In [37]:
df = pd.DataFrame(record.model_dump() for record in records)
df.head(10)

,latitude,longitude,catalog_num,scientific_name,depth_m,large_marine_ecosystem,country,observation_date,observation_year
0,29.18391,-85.68104,87,Cladopsammia,70.0,Gulf of Mexico,USA,2005-03-22,2005
1,29.18383,-85.68099,89,Cladopsammia,71.0,Gulf of Mexico,USA,2005-03-22,2005
2,29.18408,-85.68109,91,Cladopsammia,72.0,Gulf of Mexico,USA,2005-03-22,2005
3,29.18423,-85.68084,92,Cladopsammia,71.0,Gulf of Mexico,USA,2005-03-22,2005
4,29.18424,-85.68085,94,Cladopsammia,69.0,Gulf of Mexico,USA,2005-03-22,2005
5,29.18452,-85.68097,97,Cladopsammia,69.0,Gulf of Mexico,USA,2005-03-22,2005
6,29.18431,-85.68077,98,Cladopsammia,70.0,Gulf of Mexico,USA,2005-03-22,2005
7,29.18442,-85.68079,99,Cladopsammia,72.0,Gulf of Mexico,USA,2005-03-22,2005
8,29.17190,-85.69985,103,Cladopsammia,72.0,Gulf of Mexico,USA,2005-03-23,2005
9,29.17355,-85.69879,105,Cladopsammia,78.0,Gulf of Mexico,USA,2005-03-23,2005


In [ ]:
df.to_parquet("../data/interim/se_coral_records.parquet")